# 🫁 Pneumonia Detection from Chest X-rays: a 6-model benchmark with trustworthy-AI analysis

Code: https://github.com/katalnishant/pneumonia-detection

**What this notebook does** (`python -m src.pipeline`):
1. **Patient-level** re-split of the dataset (no patient in both train and validation; the official 16-image val folder is too small)
2. Two tasks: **Normal vs Pneumonia** and **Normal / Bacterial / Viral**
3. Six models: custom CNN, the original *VGG16 v1* recipe (frozen base + Flatten head), and two-phase fine-tuned **VGG16, ResNet50, DenseNet121, EfficientNetB0**
4. **Test-time augmentation** and a **top-3 ensemble**
5. **Calibration** (temperature scaling, ECE), **MC-dropout uncertainty**, **radiologist referral** of the most uncertain cases, **95% bootstrap CIs**
6. **Grad-CAM** explanations

Everything that's tuned (thresholds, temperatures, referral cut-offs, ensemble members) is chosen on the **validation** set. The test set is only used for the final numbers.

**Setup:** dataset `paultimothymooney/chest-xray-pneumonia`, Accelerator **GPU**, Internet **on** (to download ImageNet weights).

### ⚠️ Always use **Run All** (or run the cells top to bottom)
The next cell creates the project folder. If the session restarts, `/kaggle/working` is wiped, so start again from here.

In [ ]:
import os
PROJECT = '/kaggle/working/pneumonia-detection'
os.makedirs(PROJECT + '/src', exist_ok=True)
os.chdir(PROJECT)
!nvidia-smi -L
import tensorflow as tf, keras
print('TF', tf.__version__, '| Keras', keras.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 1. Source code
Each cell writes one module of the `src/` package.

In [ ]:
%%writefile /kaggle/working/pneumonia-detection/src/__init__.py
import os

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")  # hide TensorFlow C++ info/warning spam


In [ ]:
%%writefile /kaggle/working/pneumonia-detection/src/config.py
"""Central configuration. Every script reads its defaults from here."""
import os
from pathlib import Path

# ---- Paths -------------------------------------------------------------------
ROOT_DIR = Path(__file__).resolve().parent.parent
# Kaggle path of "Chest X-Ray Images (Pneumonia)" (paultimothymooney/chest-xray-pneumonia).
DEFAULT_DATA_DIR = Path(os.environ.get("PNEUMO_DATA", "/kaggle/input/chest-xray-pneumonia/chest_xray"))
ARTIFACTS_DIR = Path(os.environ.get("PNEUMO_ARTIFACTS", ROOT_DIR / "artifacts"))
CACHE_DIR = ARTIFACTS_DIR / "cache"


def task_dir(task: str) -> Path:
    return ARTIFACTS_DIR / task


def models_dir(task: str) -> Path:
    return task_dir(task) / "models"


def preds_dir(task: str) -> Path:
    return task_dir(task) / "preds"


def figures_dir(task: str) -> Path:
    return task_dir(task) / "figures"


# ---- Tasks -------------------------------------------------------------------
# binary:     NORMAL vs PNEUMONIA
# multiclass: NORMAL vs BACTERIAL vs VIRAL pneumonia (subtype parsed from the file names)
CLASS_NAMES = {
    "binary": ["NORMAL", "PNEUMONIA"],
    "multiclass": ["NORMAL", "BACTERIAL", "VIRAL"],
}

# ---- Models ------------------------------------------------------------------
MODEL_NAMES = ["cnn", "vgg16_v1", "vgg16", "resnet50", "densenet121", "efficientnetb0"]
PRETTY = {
    "cnn": "Custom CNN (from scratch)",
    "vgg16_v1": "VGG16 v1 (frozen + Flatten head)",
    "vgg16": "VGG16 (fine-tuned)",
    "resnet50": "ResNet50 (fine-tuned)",
    "densenet121": "DenseNet121 (fine-tuned)",
    "efficientnetb0": "EfficientNetB0 (fine-tuned)",
    "ensemble": "Ensemble (top-3, TTA)",
}

# ---- Data --------------------------------------------------------------------
IMG_SIZE = 224
BATCH_SIZE = 32
VAL_FRACTION = 0.15
SEED = 42

# ---- Training ----------------------------------------------------------------
CNN_EPOCHS = 30
CNN_LR = 1e-3
CNN_PATIENCE = 6

HEAD_EPOCHS = 8         # phase 1: frozen backbone, train the new head
FINETUNE_EPOCHS = 12    # phase 2: unfreeze the last block(s), small LR
HEAD_LR = 1e-3
FINETUNE_LR = 1e-5
PATIENCE = 4

V1_EPOCHS = 15          # "vgg16_v1" reproduces the original notebook: frozen base, Adam 1e-4
V1_LR = 1e-4

# ---- Analysis ----------------------------------------------------------------
ENSEMBLE_TOP_K = 3      # ensemble = top-k models ranked by VALIDATION AUC
MC_DROPOUT_SAMPLES = 30
N_BOOTSTRAP = 1000
REFERRAL_RATE = 0.10    # share of most-uncertain cases sent to a radiologist in the headline number


In [ ]:
%%writefile /kaggle/working/pneumonia-detection/src/data.py
"""Data: discovery, patient-level splitting, cached image arrays, tf.data pipelines.

* The official val/ folder has only 16 images, so train/ + val/ are merged and re-split
  BY PATIENT (StratifiedGroupKFold): no person appears in both train and validation.
  Stratification uses the 3-class label, so the binary and 3-class tasks share the
  exact same split. The official test/ folder stays untouched.
* Every image is decoded and resized ONCE and cached as uint8 (~250 MB for 224 px),
  so all models train from memory instead of re-decoding JPEGs each epoch.
"""
from __future__ import annotations

import hashlib
import json
import re
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import keras
import numpy as np
import tensorflow as tf
from keras import layers
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight

from src import config

IMG_EXTS = {".jpeg", ".jpg", ".png"}
AUTOTUNE = tf.data.AUTOTUNE


# ---------------------------------------------------------------------------
# Discovery and labels
# ---------------------------------------------------------------------------
def subtype_label(path: str) -> int | None:
    """0 = NORMAL, 1 = BACTERIAL, 2 = VIRAL (parsed from folder + file name)."""
    p = Path(path)
    if p.parent.name.upper() == "NORMAL":
        return 0
    name = p.name.lower()
    if "bacteria" in name:
        return 1
    if "virus" in name:
        return 2
    return None


def to_task_labels(sub_labels, task: str) -> np.ndarray:
    sub = np.asarray(sub_labels)
    return (sub > 0).astype("int64") if task == "binary" else sub.astype("int64")


def list_images(split_dir: Path) -> tuple[list[str], list[int]]:
    paths, subs = [], []
    for cls in ["NORMAL", "PNEUMONIA"]:
        cls_dir = split_dir / cls
        if not cls_dir.exists():
            continue
        for p in sorted(cls_dir.iterdir()):
            if p.suffix.lower() in IMG_EXTS:
                s = subtype_label(str(p))
                if s is not None:
                    paths.append(str(p))
                    subs.append(s)
    return paths, subs


def patient_id(path: str) -> str:
    """person1000_virus_1681.jpeg -> person1000 ; IM-0115-0001.jpeg -> IM-0115 ;
    NORMAL2-IM-0329-0001.jpeg -> NORMAL2-IM-0329 ; otherwise the file name."""
    name = Path(path).name
    m = re.match(r"(person\d+)_", name) or re.match(r"(.*IM-\d+)-", name)
    return f"{Path(path).parent.name}/{m.group(1) if m else Path(path).stem}"


def resolve_data_dir(data_dir) -> Path:
    data_dir = Path(data_dir)
    for cand in (data_dir, data_dir / "chest_xray"):
        if (cand / "train").exists() and (cand / "test").exists():
            return cand
    raise FileNotFoundError(f"Could not find train/ and test/ under {data_dir}")


def make_splits(data_dir=config.DEFAULT_DATA_DIR, val_fraction=config.VAL_FRACTION,
                seed=config.SEED) -> dict:
    """Returns {"train"|"val"|"test": {"paths": [...], "sub": [...] (3-class labels)}}."""
    root = resolve_data_dir(data_dir)
    tr_p, tr_s = list_images(root / "train")
    va_p, va_s = list_images(root / "val")
    pool_p, pool_s = np.array(tr_p + va_p), np.array(tr_s + va_s)
    groups = np.array([patient_id(p) for p in pool_p])

    sgkf = StratifiedGroupKFold(n_splits=max(2, round(1 / val_fraction)), shuffle=True,
                                random_state=seed)
    tr_idx, va_idx = next(sgkf.split(pool_p, pool_s, groups))
    assert not set(groups[tr_idx]) & set(groups[va_idx]), "patient leakage!"

    te_p, te_s = list_images(root / "test")
    return {
        "train": {"paths": pool_p[tr_idx].tolist(), "sub": pool_s[tr_idx].tolist()},
        "val": {"paths": pool_p[va_idx].tolist(), "sub": pool_s[va_idx].tolist()},
        "test": {"paths": te_p, "sub": te_s},
    }


def describe_splits(splits: dict) -> str:
    lines = [f"{'split':>6} {'total':>6} {'NORMAL':>7} {'BACTERIAL':>10} {'VIRAL':>6}"]
    for name, d in splits.items():
        c = np.bincount(d["sub"], minlength=3)
        lines.append(f"{name:>6} {len(d['sub']):>6} {c[0]:>7} {c[1]:>10} {c[2]:>6}")
    return "\n".join(lines)


# ---------------------------------------------------------------------------
# Image loading (shared by training and inference => identical preprocessing)
# ---------------------------------------------------------------------------
def load_gray(src, img_size: int = config.IMG_SIZE) -> np.ndarray:
    """Path or PIL image -> uint8 array (img_size, img_size), grayscale."""
    img = src if isinstance(src, Image.Image) else Image.open(src)
    return np.asarray(img.convert("L").resize((img_size, img_size), Image.BILINEAR),
                      dtype=np.uint8)


def load_split_arrays(paths: list[str], img_size: int, name: str) -> np.ndarray:
    """Decode + resize all images once; cache to disk as .npy (keyed by the path list)."""
    config.CACHE_DIR.mkdir(parents=True, exist_ok=True)
    key = hashlib.md5(json.dumps([img_size] + list(paths)).encode()).hexdigest()[:10]
    cache = config.CACHE_DIR / f"{name}_{img_size}_{key}.npy"
    if cache.exists():
        return np.load(cache)
    with ThreadPoolExecutor(max_workers=8) as ex:
        arr = np.stack(list(ex.map(lambda p: load_gray(p, img_size), paths)))
    np.save(cache, arr)
    return arr


def load_all(task: str, data_dir=config.DEFAULT_DATA_DIR, img_size=config.IMG_SIZE,
             limit: int | None = None) -> dict:
    """Return {"train"|"val"|"test": (x_uint8 [N,S,S], y [N])} for `task`.
    `limit` keeps a small stratified subset per split (for quick smoke runs)."""
    splits = make_splits(data_dir)
    out = {"paths": {}}
    for name, d in splits.items():
        paths, sub = d["paths"], np.asarray(d["sub"])
        if limit:
            rng = np.random.default_rng(config.SEED)
            idx = np.sort(rng.permutation(len(paths))[:limit])
            paths, sub = [paths[i] for i in idx], sub[idx]
        x = load_split_arrays(paths, img_size, name)
        out[name] = (x, to_task_labels(sub, task))
        out["paths"][name] = paths
    return out


def get_class_weights(y) -> dict[int, float]:
    classes = np.unique(y)
    w = compute_class_weight("balanced", classes=classes, y=y)
    return {int(c): float(v) for c, v in zip(classes, w)}


# ---------------------------------------------------------------------------
# tf.data
# ---------------------------------------------------------------------------
def build_augmenter() -> keras.Sequential:
    """Mild, anatomically plausible augmentation. No horizontal flip: the heart is on the
    left, so a mirrored chest X-ray is not a realistic image."""
    return keras.Sequential([
        layers.RandomRotation(0.03),
        layers.RandomZoom(0.1),
        layers.RandomTranslation(0.05, 0.05),
        layers.RandomContrast(0.15),
    ], name="augmentation")


def to_model_input(x_uint8):
    """uint8 (B,S,S) -> float32 (B,S,S,3) in [0,255]: the input format of every model."""
    x = tf.cast(x_uint8, tf.float32)[..., None]
    return tf.repeat(x, 3, axis=-1)


def encode_labels(y, task: str, n_classes: int):
    if task == "binary":
        return tf.cast(y, tf.float32)
    return tf.one_hot(y, n_classes)


def make_dataset(x, y, task: str, batch_size=config.BATCH_SIZE, training=False,
                 seed=config.SEED) -> tf.data.Dataset:
    n_classes = len(config.CLASS_NAMES[task])
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(len(x), seed=seed, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    aug = build_augmenter() if training else None

    def prep(xb, yb):
        xb = to_model_input(xb)
        if aug is not None:
            xb = tf.clip_by_value(aug(xb, training=True), 0.0, 255.0)
        return xb, encode_labels(yb, task, n_classes)

    return ds.map(prep, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


In [ ]:
%%writefile /kaggle/working/pneumonia-detection/src/models.py
"""Model zoo: custom CNN, the original VGG16 v1 recipe, and four fine-tuned ImageNet backbones.

All models
  * take float images in [0, 255] with shape (S, S, 3): preprocessing lives inside the model;
  * are keras.Sequential: [preprocess] -> [backbone or conv blocks] -> [head].
    Grad-CAM and MC-dropout simply walk model.layers (see gradcam.py / predict.py);
  * output sigmoid(1) for the binary task or softmax(3) for the 3-class task.
"""
from __future__ import annotations

import keras
from keras import layers

from src import config

IMAGENET_MEAN_BGR = [103.939, 116.779, 123.68]
TORCH_MEAN = [0.485 * 255, 0.456 * 255, 0.406 * 255]
TORCH_VAR = [(0.229 * 255) ** 2, (0.224 * 255) ** 2, (0.225 * 255) ** 2]

# name -> (constructor, preprocessing mode, layer-name prefixes unfrozen in phase 2)
BACKBONES = {
    "vgg16": (keras.applications.VGG16, "caffe", ("block5",)),
    "vgg16_v1": (keras.applications.VGG16, "caffe", ()),
    "resnet50": (keras.applications.ResNet50, "caffe", ("conv5",)),
    "densenet121": (keras.applications.DenseNet121, "torch", ("conv5",)),
    "efficientnetb0": (keras.applications.EfficientNetB0, "none", ("block6", "block7", "top")),
}


def preprocess_layer(mode: str):
    """Reproduces keras.applications.<net>.preprocess_input as a serialisable layer.
    'caffe' = RGB->BGR + ImageNet mean subtraction. Our X-rays have 3 identical channels,
    so the channel swap is a no-op and a fixed Normalization layer is exact.
    'torch' = scale to [0,1] then ImageNet mean/std. 'none' = model normalises internally."""
    if mode == "caffe":
        return layers.Normalization(mean=IMAGENET_MEAN_BGR, variance=[1.0] * 3, name="preprocess")
    if mode == "torch":
        return layers.Normalization(mean=TORCH_MEAN, variance=TORCH_VAR, name="preprocess")
    if mode == "rescale":
        return layers.Rescaling(1.0 / 255, name="preprocess")
    return None


def output_layer(task: str):
    if task == "binary":
        return layers.Dense(1, activation="sigmoid", name="output", dtype="float32")
    return layers.Dense(len(config.CLASS_NAMES[task]), activation="softmax", name="output",
                        dtype="float32")


def build_cnn(task: str, img_size: int) -> keras.Model:
    blocks = []
    for i, f in enumerate([32, 64, 128, 256], start=1):
        blocks += [
            layers.Conv2D(f, 3, padding="same", activation="relu",
                          name="last_conv" if i == 4 else f"conv{i}a"),
            layers.BatchNormalization(name=f"bn{i}"),
            layers.MaxPooling2D(name=f"pool{i}"),
        ]
    return keras.Sequential(
        [keras.Input((img_size, img_size, 3)), preprocess_layer("rescale")] + blocks + [
            layers.GlobalAveragePooling2D(name="gap"),
            layers.Dense(128, activation="relu", name="fc"),
            layers.Dropout(0.5, name="dropout"),
            output_layer(task)],
        name="cnn")


def build_backbone_model(name: str, task: str, img_size: int, weights="imagenet") -> keras.Model:
    ctor, mode, _ = BACKBONES[name]
    backbone = ctor(include_top=False, weights=weights, input_shape=(img_size, img_size, 3))
    pre = preprocess_layer(mode)
    if name == "vgg16_v1":
        # The original notebook: frozen VGG16 -> Flatten -> Dense(256) -> Dropout(0.5) -> out
        head = [layers.Flatten(name="flatten"), layers.Dense(256, activation="relu", name="fc"),
                layers.Dropout(0.5, name="dropout"), output_layer(task)]
    else:
        head = [layers.GlobalAveragePooling2D(name="gap"),
                layers.Dense(256, activation="relu", name="fc"),
                layers.Dropout(0.5, name="dropout"), output_layer(task)]
    stack = [keras.Input((img_size, img_size, 3))] + ([pre] if pre else []) + [backbone] + head
    return keras.Sequential(stack, name=name)


def build_model(name: str, task: str, img_size=config.IMG_SIZE, weights="imagenet"):
    if name == "cnn":
        return build_cnn(task, img_size)
    if name in BACKBONES:
        return build_backbone_model(name, task, img_size, weights)
    raise ValueError(f"unknown model {name}; choose from {config.MODEL_NAMES}")


def get_backbone(model: keras.Model):
    """The nested ImageNet backbone (a keras.Model inside the Sequential), or None."""
    for layer in model.layers:
        if isinstance(layer, keras.Model):
            return layer
    return None


def set_phase(model: keras.Model, name: str, phase: str) -> None:
    """phase 'head': whole backbone frozen. phase 'finetune': unfreeze layers whose name
    starts with the backbone's prefixes, EXCEPT BatchNorm layers, which stay frozen
    (in Keras 3 a frozen BN runs in inference mode, so its running stats are preserved)."""
    backbone = get_backbone(model)
    if backbone is None:
        return
    if phase == "head":
        backbone.trainable = False
        return
    prefixes = BACKBONES[name][2]
    backbone.trainable = True
    for layer in backbone.layers:
        layer.trainable = (layer.name.startswith(prefixes)
                           and not isinstance(layer, layers.BatchNormalization))


def compile_model(model: keras.Model, task: str, lr: float) -> None:
    n = len(config.CLASS_NAMES[task])
    if task == "binary":
        loss, acc = "binary_crossentropy", keras.metrics.BinaryAccuracy(name="accuracy")
        auc = keras.metrics.AUC(name="auc")
    else:
        loss, acc = "categorical_crossentropy", keras.metrics.CategoricalAccuracy(name="accuracy")
        auc = keras.metrics.AUC(name="auc", multi_label=True, num_labels=n)  # macro one-vs-rest
    model.compile(optimizer=keras.optimizers.Adam(lr), loss=loss, metrics=[acc, auc])


def gradcam_layer_name(model: keras.Model) -> str:
    """Grad-CAM target: the backbone's final feature map, or the CNN's last conv layer."""
    bb = get_backbone(model)
    return bb.name if bb is not None else "last_conv"


In [ ]:
%%writefile /kaggle/working/pneumonia-detection/src/train.py
"""Train one model on one task.

    python -m src.train --task binary --model densenet121
    python -m src.train --task multiclass --model efficientnetb0 --quick   # tiny smoke run
"""
from __future__ import annotations

import argparse
import json
import time

import keras
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src import config
from src.data import get_class_weights, load_all, make_dataset
from src.models import build_model, compile_model, set_phase


def callbacks(path, patience, best_so_far=None):
    return [
        keras.callbacks.ModelCheckpoint(path, monitor="val_auc", mode="max", save_best_only=True,
                                        initial_value_threshold=best_so_far, verbose=0),
        keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=patience,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2,
                                          min_lr=1e-7, verbose=1),
    ]


def merge(*hists) -> dict:
    out: dict = {}
    for h in hists:
        for k, v in h.history.items():
            out.setdefault(k, []).extend(float(x) for x in v)
    return out


def plot_history(hist, name, task, boundary=None):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, m in zip(axes, ["loss", "accuracy", "auc"]):
        ax.plot(hist[m], label="train")
        ax.plot(hist[f"val_{m}"], label="val")
        if boundary:
            ax.axvline(boundary - 0.5, color="gray", ls="--", label="fine-tuning starts")
        ax.set(title=m, xlabel="epoch")
        ax.grid(alpha=0.3)
    axes[0].legend()
    fig.suptitle(f"{config.PRETTY[name]} - {task}")
    fig.tight_layout()
    config.figures_dir(task).mkdir(parents=True, exist_ok=True)
    fig.savefig(config.figures_dir(task) / f"history_{name}.png", dpi=110)
    plt.close(fig)


def train_model(name: str, task: str, data: dict, img_size=config.IMG_SIZE,
                batch_size=config.BATCH_SIZE, weights="imagenet", epoch_scale: float = 1.0,
                seed=config.SEED) -> dict:
    """Train `name` on `task` using pre-loaded `data` (from load_all). Returns metadata."""
    keras.utils.set_random_seed(seed)
    mdir = config.models_dir(task)
    mdir.mkdir(parents=True, exist_ok=True)
    path = mdir / f"{name}.keras"
    ep = lambda n: max(1, round(n * epoch_scale))

    x_tr, y_tr = data["train"]
    train_ds = make_dataset(x_tr, y_tr, task, batch_size, training=True, seed=seed)
    val_ds = make_dataset(*data["val"], task, batch_size)
    cw = get_class_weights(y_tr)
    model = build_model(name, task, img_size, weights)
    t0, boundary = time.time(), None

    if name == "cnn":
        compile_model(model, task, config.CNN_LR)
        h = model.fit(train_ds, validation_data=val_ds, epochs=ep(config.CNN_EPOCHS),
                      class_weight=cw, callbacks=callbacks(path, config.CNN_PATIENCE), verbose=2)
        hist = merge(h)
    elif name == "vgg16_v1":
        # Original recipe: frozen base, Adam(1e-4), no class weights, no fine-tuning.
        set_phase(model, name, "head")
        compile_model(model, task, config.V1_LR)
        h = model.fit(train_ds, validation_data=val_ds, epochs=ep(config.V1_EPOCHS),
                      callbacks=callbacks(path, config.PATIENCE), verbose=2)
        hist = merge(h)
    else:
        set_phase(model, name, "head")
        compile_model(model, task, config.HEAD_LR)
        h1 = model.fit(train_ds, validation_data=val_ds, epochs=ep(config.HEAD_EPOCHS),
                       class_weight=cw, callbacks=callbacks(path, config.PATIENCE), verbose=2)
        boundary = len(h1.history["loss"])
        set_phase(model, name, "finetune")
        compile_model(model, task, config.FINETUNE_LR)   # recompile after changing trainable
        h2 = model.fit(train_ds, validation_data=val_ds, initial_epoch=boundary,
                       epochs=boundary + ep(config.FINETUNE_EPOCHS), class_weight=cw,
                       callbacks=callbacks(path, config.PATIENCE, max(h1.history["val_auc"])),
                       verbose=2)
        hist = merge(h1, h2)

    meta = {"model": name, "task": task, "img_size": img_size,
            "class_names": config.CLASS_NAMES[task], "best_val_auc": max(hist["val_auc"]),
            "epochs_run": len(hist["loss"]), "train_minutes": round((time.time() - t0) / 60, 1),
            "params": int(model.count_params())}
    (mdir / f"{name}.json").write_text(json.dumps(meta, indent=2))
    (mdir / f"history_{name}.json").write_text(json.dumps(hist))
    plot_history(hist, name, task, boundary)
    print(f"[{task}/{name}] best val AUC {meta['best_val_auc']:.4f} in {meta['train_minutes']} min")
    return meta


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--task", choices=list(config.CLASS_NAMES), default="binary")
    p.add_argument("--model", choices=config.MODEL_NAMES, required=True)
    p.add_argument("--data_dir", default=str(config.DEFAULT_DATA_DIR))
    p.add_argument("--img_size", type=int, default=config.IMG_SIZE)
    p.add_argument("--weights", default="imagenet", help="'imagenet' or 'none'")
    p.add_argument("--quick", action="store_true", help="tiny subset + 1-2 epochs (smoke test)")
    a = p.parse_args()
    data = load_all(a.task, a.data_dir, a.img_size, limit=200 if a.quick else None)
    train_model(a.model, a.task, data, a.img_size,
                weights=None if a.weights == "none" else a.weights,
                epoch_scale=0.1 if a.quick else 1.0)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/pneumonia-detection/src/predict.py
"""Inference for a trained model on val + test: plain, test-time augmentation (TTA) and
Monte-Carlo dropout. Results go to artifacts/<task>/preds/<model>.npz for analysis.

    python -m src.predict --task binary --model densenet121
"""
from __future__ import annotations

import argparse

import keras
import numpy as np
import tensorflow as tf

from src import config
from src.data import load_all, to_model_input

# TTA views as crop boxes (y1, x1, y2, x2) in relative coords, each resized back to full size:
# identity, 8% centre zoom, and small shifts. All are plausible re-positionings of a patient.
TTA_BOXES = np.array([
    [0.00, 0.00, 1.00, 1.00],
    [0.04, 0.04, 0.96, 0.96],
    [0.00, 0.00, 0.94, 0.94],
    [0.06, 0.06, 1.00, 1.00],
    [0.00, 0.06, 0.94, 1.00],
], dtype="float32")


def as_matrix(p: np.ndarray, task: str) -> np.ndarray:
    """Binary sigmoid output (N,1) -> (N,2) class-probability matrix; softmax stays (N,C)."""
    p = np.asarray(p, dtype="float64")
    if task == "binary":
        p = p.reshape(-1)
        return np.stack([1 - p, p], axis=1)
    return p


def _batches(x, batch_size=64):
    return tf.data.Dataset.from_tensor_slices(x).batch(batch_size).map(to_model_input)


def predict_plain(model, x) -> np.ndarray:
    return model.predict(_batches(x), verbose=0)


def predict_tta(model, x) -> np.ndarray:
    size = model.input_shape[1:3]
    preds = []
    for box in TTA_BOXES:
        def view(xb, box=box):
            n = tf.shape(xb)[0]
            boxes = tf.tile(box[None], [n, 1])
            return tf.image.crop_and_resize(xb, boxes, tf.range(n), size)
        ds = _batches(x).map(view)
        preds.append(model.predict(ds, verbose=0))
    return np.mean(preds, axis=0)


def split_head(model: keras.Model):
    """Split a model into (feature extractor, head) at the pooling layer (gap / flatten)."""
    names = [l.name for l in model.layers]
    k = names.index("gap") if "gap" in names else names.index("flatten")
    def chain(layers, shape):
        inp = keras.Input(shape)
        h = inp
        for layer in layers:
            h = layer(h)
        return keras.Model(inp, h)

    feat = chain(model.layers[:k + 1], model.input_shape[1:])
    head = chain(model.layers[k + 1:], tuple(feat.output.shape[1:]))
    return feat, head


def predict_mc_dropout(model, x, n_samples=config.MC_DROPOUT_SAMPLES, seed=config.SEED):
    """MC dropout on the classifier head: features are computed once, then the head is run
    `n_samples` times with dropout ACTIVE. Returns (mean probs, samples [T, N, out])."""
    keras.utils.set_random_seed(seed)
    feat, head = split_head(model)
    f = feat.predict(_batches(x), verbose=0)
    samples = np.stack([np.asarray(head(f, training=True)) for _ in range(n_samples)])
    return samples.mean(axis=0), samples


def run(task: str, name: str, data: dict) -> dict:
    model = keras.models.load_model(config.models_dir(task) / f"{name}.keras")
    out = {}
    for split in ["val", "test"]:
        x, y = data[split]
        out[f"y_{split}"] = y
        out[f"{split}_plain"] = as_matrix(predict_plain(model, x), task)
        out[f"{split}_tta"] = as_matrix(predict_tta(model, x), task)
        mean, samples = predict_mc_dropout(model, x)
        out[f"{split}_mc"] = as_matrix(mean, task)
        # Uncertainty = std of the predicted-class probability across MC samples.
        s = np.stack([as_matrix(t, task) for t in samples])          # (T, N, C)
        pred = out[f"{split}_mc"].argmax(1)
        out[f"{split}_mc_std"] = s[:, np.arange(len(pred)), pred].std(axis=0)
    config.preds_dir(task).mkdir(parents=True, exist_ok=True)
    np.savez_compressed(config.preds_dir(task) / f"{name}.npz", **out)
    keras.backend.clear_session()
    return out


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--task", choices=list(config.CLASS_NAMES), default="binary")
    p.add_argument("--model", choices=config.MODEL_NAMES, required=True)
    p.add_argument("--data_dir", default=str(config.DEFAULT_DATA_DIR))
    a = p.parse_args()
    meta_size = keras.models.load_model(config.models_dir(a.task) / f"{a.model}.keras").input_shape[1]
    run(a.task, a.model, load_all(a.task, a.data_dir, meta_size))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/pneumonia-detection/src/analyze.py
"""Analysis of all trained models for one task (run after src.predict for every model).

For every model and for a top-k ensemble:
  * test-time augmentation (TTA) vs plain predictions
  * temperature scaling fitted on VALIDATION -> calibrated probabilities, ECE before/after
  * binary: decision threshold tuned on validation (Youden's J)
  * test metrics with 95% bootstrap confidence intervals
  * uncertainty-based referral: the most uncertain cases (threshold chosen on validation)
    are routed to a radiologist; we report accuracy on the cases the model keeps
Outputs: artifacts/<task>/results.json, results.md and figures/*.png

    python -m src.analyze --task binary
"""
from __future__ import annotations

import argparse
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize_scalar
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay)

from src import config

# Reference categorical palette (fixed order, validated for colour-vision deficiency).
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]
NEUTRAL, INK, MUTED = "#b8b7b0", "#0b0b0b", "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.titleweight": "bold", "font.size": 10})


# ---------------------------------------------------------------------------
# Metric helpers
# ---------------------------------------------------------------------------
def auc(y, P):
    if P.shape[1] == 2:
        return roc_auc_score(y, P[:, 1])
    return roc_auc_score(y, P, multi_class="ovr", average="macro")


def ece(y, P, n_bins=15) -> float:
    """Expected calibration error of the top-class confidence."""
    conf, pred = P.max(1), P.argmax(1)
    correct = (pred == y).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    total = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(total)


def fit_temperature(y, P) -> float:
    logp = np.log(np.clip(P, 1e-7, 1))

    def nll(t):
        z = logp / t
        z -= z.max(1, keepdims=True)
        lp = z - np.log(np.exp(z).sum(1, keepdims=True))
        return -lp[np.arange(len(y)), y].mean()
    return float(minimize_scalar(nll, bounds=(0.05, 10), method="bounded").x)


def apply_temperature(P, t):
    z = np.log(np.clip(P, 1e-7, 1)) / t
    z -= z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def youden(y, p) -> float:
    fpr, tpr, thr = roc_curve(y, p)
    return float(np.clip(thr[np.argmax(tpr - fpr)], 0, 1))


def predict_labels(P, thr):
    return (P[:, 1] >= thr).astype(int) if P.shape[1] == 2 else P.argmax(1)


def point_metrics(y, P, thr) -> dict:
    yp = predict_labels(P, thr)
    if P.shape[1] == 2:
        tn, fp, fn, tp = confusion_matrix(y, yp, labels=[0, 1]).ravel()
        return {"accuracy": accuracy_score(y, yp), "auc": auc(y, P),
                "f1": f1_score(y, yp), "precision": precision_score(y, yp, zero_division=0),
                "sensitivity": recall_score(y, yp), "specificity": tn / (tn + fp)}
    rec = recall_score(y, yp, average=None, labels=list(range(P.shape[1])), zero_division=0)
    out = {"accuracy": accuracy_score(y, yp), "auc": auc(y, P),
           "f1": f1_score(y, yp, average="macro")}
    out.update({f"recall_{c.lower()}": float(r)
                for c, r in zip(config.CLASS_NAMES["multiclass"], rec)})
    return out


def bootstrap_ci(y, P, thr, n=config.N_BOOTSTRAP, seed=config.SEED) -> dict:
    rng = np.random.default_rng(seed)
    keys = ["accuracy", "auc", "f1"]
    samples = {k: [] for k in keys}
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        if len(np.unique(y[idx])) < P.shape[1]:
            continue
        m = point_metrics(y[idx], P[idx], thr)
        for k in keys:
            samples[k].append(m[k])
    return {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]
            for k, v in samples.items()}


def referral(y_val, P_val, u_val, y_te, P_te, u_te, thr, rate=config.REFERRAL_RATE) -> dict:
    """Refer cases with uncertainty above the validation (1-rate) quantile."""
    cut = np.quantile(u_val, 1 - rate)
    keep = u_te <= cut
    yp = predict_labels(P_te, thr)
    correct = yp == y_te
    return {"uncertainty_cutoff": float(cut), "referred_fraction": float(1 - keep.mean()),
            "accuracy_all": float(correct.mean()), "accuracy_kept": float(correct[keep].mean()),
            "errors_caught": int((~correct & ~keep).sum()), "errors_total": int((~correct).sum())}


def coverage_curve(y, P, u, thr, coverages=np.linspace(0.5, 1.0, 26)):
    order = np.argsort(u)                        # most certain first
    correct = (predict_labels(P, thr) == y)[order]
    return coverages, [correct[: max(1, int(round(c * len(y))))].mean() for c in coverages]


# ---------------------------------------------------------------------------
def evaluate_model(name, d, task) -> dict:
    """d holds y_val, y_test and val/test probability matrices: *_plain, *_tta, *_unc."""
    y_val, y_te = d["y_val"], d["y_test"]
    t = fit_temperature(y_val, d["val_tta"])
    Pv, Pt = apply_temperature(d["val_tta"], t), apply_temperature(d["test_tta"], t)
    thr = youden(y_val, Pv[:, 1]) if task == "binary" else None
    res = {
        "name": name, "pretty": config.PRETTY[name],
        "val_auc": auc(y_val, d["val_tta"]),
        "test_auc_plain": auc(y_te, d["test_plain"]), "test_auc_tta": auc(y_te, d["test_tta"]),
        "temperature": t, "threshold": thr,
        "ece_before": ece(y_te, d["test_tta"]), "ece_after": ece(y_te, Pt),
        "test": point_metrics(y_te, Pt, thr),
        "test_at_0.5": point_metrics(y_te, d["test_plain"], 0.5) if task == "binary" else None,
        "ci95": bootstrap_ci(y_te, Pt, thr),
        "referral": referral(y_val, Pv, d["val_unc"], y_te, Pt, d["test_unc"], thr),
    }
    res["_P_test"], res["_P_val"] = Pt, Pv
    return res


def build_ensemble(preds: dict, ranked: list[str], k: int) -> dict:
    members = ranked[:k]
    d = {"y_val": preds[members[0]]["y_val"], "y_test": preds[members[0]]["y_test"]}
    for s in ["val", "test"]:
        stack = np.stack([preds[m][f"{s}_tta"] for m in members])         # (K, N, C)
        d[f"{s}_tta"] = stack.mean(0)
        d[f"{s}_plain"] = np.stack([preds[m][f"{s}_plain"] for m in members]).mean(0)
        pred = d[f"{s}_tta"].argmax(1)
        # Uncertainty = disagreement between members on the predicted class.
        d[f"{s}_unc"] = stack[:, np.arange(stack.shape[1]), pred].std(0)
    d["members"] = members
    return d


# ---------------------------------------------------------------------------
# Figures
# ---------------------------------------------------------------------------
def fig_comparison(results, task, path):
    names = [r["name"] for r in results]
    fig, axes = plt.subplots(1, 2, figsize=(13, 0.55 * len(names) + 1.8), sharey=True)
    best = max(results, key=lambda r: r["val_auc"])["name"]
    for ax, key, label in [(axes[0], "auc", "Test ROC-AUC"),
                           (axes[1], "f1", "Test F1" if task == "binary" else "Test macro-F1")]:
        vals = [r["test"][key] for r in results]
        lo = [r["test"][key] - r["ci95"][key][0] for r in results]
        hi = [r["ci95"][key][1] - r["test"][key] for r in results]
        colors = [SERIES[0] if n in ("ensemble", best) else NEUTRAL for n in names]
        y = np.arange(len(names))
        ax.barh(y, vals, color=colors, height=0.6, xerr=[lo, hi], ecolor=MUTED, capsize=3)
        for yi, v in zip(y, vals):
            ax.text(0.505, yi, f"{v:.3f}", va="center", ha="left", color="white", fontsize=9,
                    fontweight="bold")
        ax.set_xlim(0.5, 1.0)
        ax.set_title(label)
        ax.grid(axis="x", alpha=0.25)
    axes[0].set_yticks(np.arange(len(names)), [config.PRETTY[n] for n in names])
    axes[0].invert_yaxis()
    fig.suptitle(f"Model comparison on the held-out test set ({task}) - bars show 95% bootstrap CI",
                 fontsize=11)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


def fig_roc(results, y_te, task, path):
    show = ["cnn", "vgg16_v1", max([r for r in results if r["name"] != "ensemble"],
                                    key=lambda r: r["val_auc"])["name"], "ensemble"]
    show = list(dict.fromkeys(n for n in show if any(r["name"] == n for r in results)))
    fig, ax = plt.subplots(figsize=(6, 6))
    for i, n in enumerate(show):
        r = next(r for r in results if r["name"] == n)
        P = r["_P_test"]
        if task == "binary":
            fpr, tpr, _ = roc_curve(y_te, P[:, 1])
        else:  # micro-average one-vs-rest
            Y = np.eye(P.shape[1])[y_te]
            fpr, tpr, _ = roc_curve(Y.ravel(), P.ravel())
        ax.plot(fpr, tpr, color=SERIES[i], lw=2, label=f"{config.PRETTY[n]} (AUC {r['test']['auc']:.3f})")
    ax.plot([0, 1], [0, 1], color=NEUTRAL, ls="--", lw=1)
    ax.set(xlabel="False positive rate", ylabel="True positive rate",
           title=f"ROC curves - test set ({task})")
    ax.legend(loc="lower right", fontsize=8, frameon=False)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


def fig_confusion(r, y_te, task, path):
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    yp = predict_labels(r["_P_test"], r["threshold"])
    n = len(config.CLASS_NAMES[task])
    ConfusionMatrixDisplay(confusion_matrix(y_te, yp, labels=list(range(n))),
                           display_labels=config.CLASS_NAMES[task]).plot(
        ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{config.PRETTY[r['name']]}", fontsize=10)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


def fig_reliability(r, d, path, n_bins=10):
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    bins = np.linspace(0, 1, n_bins + 1)
    for i, (P, label) in enumerate([(d["test_tta"], f"before (ECE {r['ece_before']:.3f})"),
                                    (r["_P_test"], f"after temperature scaling (ECE {r['ece_after']:.3f})")]):
        conf, correct = P.max(1), P.argmax(1) == d["y_test"]
        xs, ys = [], []
        for lo, hi in zip(bins[:-1], bins[1:]):
            m = (conf > lo) & (conf <= hi)
            if m.sum() >= 5:
                xs.append(conf[m].mean())
                ys.append(correct[m].mean())
        ax.plot(xs, ys, marker="o", ms=6, lw=2, color=SERIES[i], label=label)
    ax.plot([0, 1], [0, 1], color=NEUTRAL, ls="--", lw=1, label="perfect calibration")
    ax.set(xlabel="Predicted confidence", ylabel="Observed accuracy", xlim=(0.4, 1), ylim=(0.4, 1),
           title=f"Reliability - {config.PRETTY[r['name']]}")
    ax.legend(loc="upper left", fontsize=8, frameon=False)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


def fig_referral(curves, path):
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for i, (label, cov, acc) in enumerate(curves):
        baseline = label.startswith("Random")
        ax.plot(cov * 100, np.array(acc) * 100, lw=1.5 if baseline else 2,
                ls="--" if baseline else "-", color=NEUTRAL if baseline else SERIES[i], label=label)
    ax.axvline(100 * (1 - config.REFERRAL_RATE), color=NEUTRAL, ls="--", lw=1)
    ax.text(100 * (1 - config.REFERRAL_RATE) - 0.5, ax.get_ylim()[0] + 0.3,
            f"refer {config.REFERRAL_RATE:.0%}", ha="right", color=MUTED, fontsize=8)
    ax.invert_xaxis()
    ax.set(xlabel="Coverage: % of test cases the model decides (rest referred to a radiologist)",
           ylabel="Accuracy on decided cases (%)", title="Uncertainty-based referral")
    ax.legend(frameon=False, fontsize=8)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


# ---------------------------------------------------------------------------
def to_markdown(results, task, ens_members) -> str:
    f1 = "F1" if task == "binary" else "Macro-F1"
    head = f"| Model | ROC-AUC [95% CI] | Accuracy [95% CI] | {f1} [95% CI] |"
    extra = " Sensitivity | Specificity |" if task == "binary" else \
        " Recall normal | Recall bacterial | Recall viral |"
    lines = [head + extra + " ECE (before -> after) |",
             "|---|---|---|---|" + "---|" * (2 if task == "binary" else 3) + "---|"]
    for r in results:
        t, ci = r["test"], r["ci95"]
        cells = [f"{t[k]:.3f} [{ci[k][0]:.3f}-{ci[k][1]:.3f}]" for k in ["auc", "accuracy", "f1"]]
        if task == "binary":
            cells += [f"{t['sensitivity']:.3f}", f"{t['specificity']:.3f}"]
        else:
            cells += [f"{t[f'recall_{c}']:.3f}" for c in ["normal", "bacterial", "viral"]]
        cells.append(f"{r['ece_before']:.3f} -> {r['ece_after']:.3f}")
        name = f"**{r['pretty']}**" if r["name"] == "ensemble" else r["pretty"]
        lines.append(f"| {name} | " + " | ".join(cells) + " |")
    lines += ["", f"Ensemble members (top-{len(ens_members)} by validation AUC): "
              + ", ".join(config.PRETTY[m] for m in ens_members), "",
              "| Model | Test AUC plain | Test AUC with TTA | Referral: accuracy all -> kept "
              f"(refer {config.REFERRAL_RATE:.0%}) | Errors caught by referral |", "|---|---|---|---|---|"]
    for r in results:
        rf = r["referral"]
        lines.append(f"| {r['pretty']} | {r['test_auc_plain']:.3f} | {r['test_auc_tta']:.3f} | "
                     f"{rf['accuracy_all']:.3f} -> {rf['accuracy_kept']:.3f} "
                     f"({rf['referred_fraction']:.1%} referred) | "
                     f"{rf['errors_caught']}/{rf['errors_total']} |")
    return "\n".join(lines) + "\n"


def main(task: str):
    pdir, fdir = config.preds_dir(task), config.figures_dir(task)
    fdir.mkdir(parents=True, exist_ok=True)
    preds = {}
    for n in config.MODEL_NAMES:
        f = pdir / f"{n}.npz"
        if f.exists():
            z = dict(np.load(f))
            z["val_unc"], z["test_unc"] = z["val_mc_std"], z["test_mc_std"]
            preds[n] = z
    if not preds:
        raise SystemExit("no predictions found - run src.predict first")

    results = [evaluate_model(n, d, task) for n, d in preds.items()]
    ranked = [r["name"] for r in sorted(results, key=lambda r: -r["val_auc"])]
    k = min(config.ENSEMBLE_TOP_K, len(ranked))
    ens = build_ensemble(preds, ranked, k)
    results.append(evaluate_model("ensemble", ens, task))
    y_te = ens["y_test"]
    best_single = ranked[0]

    # Figures
    fig_comparison(results, task, fdir / "model_comparison.png")
    fig_roc(results, y_te, task, fdir / "roc_curves.png")
    for n in [best_single, "ensemble"]:
        fig_confusion(next(r for r in results if r["name"] == n), y_te, task,
                      fdir / f"confusion_{n}.png")
    best_r = next(r for r in results if r["name"] == best_single)
    fig_reliability(best_r, preds[best_single], fdir / "reliability.png")
    curves = []
    for n, d, lbl in [("ensemble", ens, "Ensemble - member disagreement"),
                      (best_single, preds[best_single], f"{config.PRETTY[best_single]} - MC dropout")]:
        r = next(r for r in results if r["name"] == n)
        cov, acc = coverage_curve(y_te, r["_P_test"], d["test_unc"], r["threshold"])
        curves.append((lbl, cov, acc))
    rng = np.random.default_rng(0)
    cov, acc = coverage_curve(y_te, best_r["_P_test"], rng.random(len(y_te)), best_r["threshold"])
    curves.append(("Random referral (baseline)", cov, acc))
    fig_referral(curves, fdir / "referral_curve.png")

    # Tables
    md = to_markdown(results, task, ens["members"])
    (config.task_dir(task) / "results.md").write_text(md)
    clean = [{k: v for k, v in r.items() if not k.startswith("_")} for r in results]
    summary = {"task": task, "class_names": config.CLASS_NAMES[task], "n_test": int(len(y_te)),
               "best_single_model": best_single, "ensemble_members": ens["members"],
               "models": clean}
    (config.task_dir(task) / "results.json").write_text(json.dumps(summary, indent=2, default=float))
    print(md)


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--task", choices=list(config.CLASS_NAMES), default="binary")
    main(p.parse_args().task)


In [ ]:
%%writefile /kaggle/working/pneumonia-detection/src/gradcam.py
"""Grad-CAM (Selvaraju et al., ICCV 2017) for any model in the zoo.

Models are flat Sequential stacks ([preprocess] -> backbone | conv blocks -> head), so we
run the layers one by one inside a GradientTape and grab the target feature map.

    python -m src.gradcam --task binary --model densenet121
"""
from __future__ import annotations

import argparse

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from src import config
from src.models import gradcam_layer_name


def gradcam_heatmap(model, x: np.ndarray, class_idx: int | None = None):
    """x: (1, S, S, 3) float in [0,255]. Returns (heatmap [S,S] in [0,1], probs [C])."""
    target = gradcam_layer_name(model)
    h = tf.convert_to_tensor(x, tf.float32)
    fmap = None
    with tf.GradientTape() as tape:
        for layer in model.layers:
            h = layer(h, training=False)
            if layer.name == target:
                fmap = h
                tape.watch(fmap)       # works even when the layer is frozen
        probs = h[0]
        if probs.shape[-1] == 1:       # binary sigmoid: explain the PNEUMONIA score
            score = probs[0]
        else:
            idx = int(tf.argmax(probs)) if class_idx is None else class_idx
            score = probs[idx]
    grads = tape.gradient(score, fmap)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.nn.relu(tf.reduce_sum(fmap[0] * weights, axis=-1))
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], x.shape[1:3], method="bilinear")[..., 0]
    p = probs.numpy()
    p = np.array([1 - p[0], p[0]]) if p.shape[-1] == 1 else p
    return cam.numpy(), p


def overlay(x: np.ndarray, heat: np.ndarray, alpha: float = 0.4) -> np.ndarray:
    img = x[0].astype("float32") / 255.0
    colored = matplotlib.colormaps["jet"](heat)[..., :3]
    return (np.clip((1 - alpha) * img + alpha * colored, 0, 1) * 255).astype("uint8")


def gallery(task: str, name: str, x_test, y_test, n_per_class=3, seed=config.SEED, out=None,
            threshold: float | None = None):
    """Grid of test images: correct and incorrect examples per class, with Grad-CAM."""
    import keras
    model = keras.models.load_model(config.models_dir(task) / f"{name}.keras")
    classes = config.CLASS_NAMES[task]
    rng = np.random.default_rng(seed)
    picks = []
    for c in range(len(classes)):
        idx = np.where(y_test == c)[0]
        picks += list(rng.choice(idx, min(n_per_class, len(idx)), replace=False))
    cols = len(picks)
    fig, axes = plt.subplots(2, cols, figsize=(2.3 * cols, 5.2))
    for j, i in enumerate(picks):
        x = np.repeat(x_test[i][None, ..., None].astype("float32"), 3, -1)
        heat, p = gradcam_heatmap(model, x)
        pred = int(p[1] >= threshold) if (threshold is not None and len(p) == 2) else int(p.argmax())
        axes[0, j].imshow(x_test[i], cmap="gray")
        axes[0, j].set_title(f"true: {classes[y_test[i]]}", fontsize=8)
        axes[1, j].imshow(overlay(x, heat))
        axes[1, j].set_title(f"pred: {classes[pred]} {p[pred]:.2f}", fontsize=8,
                             color="#008300" if pred == y_test[i] else "#e34948")
        for ax in axes[:, j]:
            ax.axis("off")
    fig.suptitle(f"Grad-CAM - {config.PRETTY[name]} ({task})", fontsize=11)
    fig.tight_layout()
    out = out or config.figures_dir(task) / f"gradcam_{name}.png"
    fig.savefig(out, dpi=110)
    plt.close(fig)
    return out


if __name__ == "__main__":
    from src.data import load_all
    p = argparse.ArgumentParser()
    p.add_argument("--task", choices=list(config.CLASS_NAMES), default="binary")
    p.add_argument("--model", choices=config.MODEL_NAMES, required=True)
    p.add_argument("--data_dir", default=str(config.DEFAULT_DATA_DIR))
    a = p.parse_args()
    d = load_all(a.task, a.data_dir)
    print(gallery(a.task, a.model, *d["test"]))


In [ ]:
%%writefile /kaggle/working/pneumonia-detection/src/inference.py
"""Single-image inference with everything the study learned on validation data:
calibrated probabilities (temperature), tuned threshold, MC-dropout uncertainty and the
referral rule. Used by the Streamlit app and as a CLI.

    python -m src.inference path/to/xray.jpeg --task binary
"""
from __future__ import annotations

import json
from dataclasses import dataclass

import keras
import numpy as np

from src import config
from src.analyze import apply_temperature
from src.data import load_gray
from src.predict import as_matrix, split_head


@dataclass
class Bundle:
    task: str
    name: str
    model: keras.Model
    temperature: float
    threshold: float | None
    referral_cutoff: float
    feat: keras.Model
    head: keras.Model

    @property
    def img_size(self):
        return self.model.input_shape[1]

    @property
    def classes(self):
        return config.CLASS_NAMES[self.task]


def load_bundle(task: str, name: str | None = None) -> Bundle:
    res = json.loads((config.task_dir(task) / "results.json").read_text())
    name = name or res["best_single_model"]
    m = next(r for r in res["models"] if r["name"] == name)
    model = keras.models.load_model(config.models_dir(task) / f"{name}.keras")
    feat, head = split_head(model)
    return Bundle(task, name, model, m["temperature"], m["threshold"],
                  m["referral"]["uncertainty_cutoff"], feat, head)


def to_input(img, img_size: int) -> np.ndarray:
    """PIL image or path -> (1, S, S, 3) float32 in [0,255] (same as training)."""
    g = load_gray(img, img_size).astype("float32")
    return np.repeat(g[None, ..., None], 3, axis=-1)


def predict(b: Bundle, x: np.ndarray, mc_samples: int = config.MC_DROPOUT_SAMPLES) -> dict:
    f = b.feat(x, training=False)
    samples = np.stack([as_matrix(np.asarray(b.head(f, training=True)), b.task)
                        for _ in range(mc_samples)])[:, 0]          # (T, C)
    probs = apply_temperature(samples.mean(0, keepdims=True), b.temperature)[0]
    if b.task == "binary":
        pred = int(probs[1] >= b.threshold)
    else:
        pred = int(probs.argmax())
    unc = float(samples[:, pred].std())
    return {"label": b.classes[pred], "pred": pred,
            "probs": {c: float(p) for c, p in zip(b.classes, probs)},
            "uncertainty": unc, "refer": unc > b.referral_cutoff}


if __name__ == "__main__":
    import argparse
    p = argparse.ArgumentParser()
    p.add_argument("image")
    p.add_argument("--task", choices=list(config.CLASS_NAMES), default="binary")
    p.add_argument("--model", default=None, help="default: best model of the study")
    a = p.parse_args()
    b = load_bundle(a.task, a.model)
    r = predict(b, to_input(a.image, b.img_size))
    print(json.dumps({"model": b.name, **r}, indent=2))


In [ ]:
%%writefile /kaggle/working/pneumonia-detection/src/pipeline.py
"""Run the whole study end to end: EDA -> train every model -> predict (plain, TTA,
MC dropout) -> analysis (calibration, ensemble, CIs, referral) -> Grad-CAM galleries.

    python -m src.pipeline                                   # both tasks, all models
    python -m src.pipeline --tasks binary --models cnn densenet121
    python -m src.pipeline --quick                           # 5-minute smoke run
Already-trained models are skipped unless --force is given, so a crashed run resumes.
"""
from __future__ import annotations

import argparse
import gc
import json
import time

import keras
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from src import analyze, config, gradcam, predict
from src.data import describe_splits, load_all, make_splits
from src.train import train_model


def eda(data_dir, out_dir):
    """Dataset overview figure: class counts per split + example images per class."""
    splits = make_splits(data_dir)
    print(describe_splits(splits))
    out_dir.mkdir(parents=True, exist_ok=True)
    names = config.CLASS_NAMES["multiclass"]
    colors = ["#2a78d6", "#eb6834", "#1baf7a"]
    fig = plt.figure(figsize=(14, 6.5))
    ax = fig.add_subplot(1, 4, 1)
    x = np.arange(3)
    for i, c in enumerate(names):
        counts = [np.sum(np.asarray(splits[s]["sub"]) == i) for s in ["train", "val", "test"]]
        ax.bar(x + (i - 1) * 0.27, counts, 0.25, color=colors[i], label=c)
    ax.set_xticks(x, ["train", "val", "test"])
    ax.set_title("Images per class", fontweight="bold")
    ax.legend(frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    tr = splits["train"]
    for i, c in enumerate(names):
        idx = [j for j, s in enumerate(tr["sub"]) if s == i][:2]
        for k, j in enumerate(idx):
            a = fig.add_subplot(2, 4, 2 + i + 4 * k)
            a.imshow(plt.imread(tr["paths"][j]), cmap="gray")
            a.set_title(c, fontsize=9)
            a.axis("off")
    fig.suptitle("Chest X-Ray Images (Pneumonia) - patient-level split", fontweight="bold")
    fig.tight_layout()
    fig.savefig(out_dir / "dataset_overview.png", dpi=110)
    plt.close(fig)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--tasks", nargs="+", default=["binary", "multiclass"],
                   choices=list(config.CLASS_NAMES))
    p.add_argument("--models", nargs="+", default=config.MODEL_NAMES, choices=config.MODEL_NAMES)
    p.add_argument("--data_dir", default=str(config.DEFAULT_DATA_DIR))
    p.add_argument("--img_size", type=int, default=config.IMG_SIZE)
    p.add_argument("--weights", default="imagenet")
    p.add_argument("--quick", action="store_true", help="small subset, ~10%% epochs")
    p.add_argument("--force", action="store_true", help="retrain models that already exist")
    a = p.parse_args()
    weights = None if a.weights == "none" else a.weights
    t_start = time.time()

    eda(a.data_dir, config.ARTIFACTS_DIR)
    for task in a.tasks:
        print(f"\n################ TASK: {task} ################")
        data = load_all(task, a.data_dir, a.img_size, limit=300 if a.quick else None)
        for name in a.models:
            ckpt = config.models_dir(task) / f"{name}.keras"
            if ckpt.exists() and not a.force:
                print(f"[{task}/{name}] already trained - skipping")
            else:
                print(f"\n======== [{task}] training {config.PRETTY[name]} ========")
                train_model(name, task, data, a.img_size, weights=weights,
                            epoch_scale=0.1 if a.quick else 1.0)
            keras.backend.clear_session()
            gc.collect()
            predict.run(task, name, data)
            gc.collect()

        analyze.main(task)
        res = json.loads((config.task_dir(task) / "results.json").read_text())
        best = res["best_single_model"]
        thr = {m["name"]: m["threshold"] for m in res["models"]}
        for name in dict.fromkeys([best, "cnn"]):
            if name in a.models:
                gradcam.gallery(task, name, *data["test"], threshold=thr[name])
        del data
        gc.collect()
    print(f"\nAll done in {(time.time() - t_start) / 60:.1f} min. Artifacts: {config.ARTIFACTS_DIR}")


if __name__ == "__main__":
    main()


## 2. Run the study
Set `QUICK = True` for a ~10-minute test on a small subset. Set it back to `False` for the real run (about 3–4 h on a T4). Already-trained models are skipped, so re-running resumes after an interruption.

In [ ]:
QUICK = False   # True = quick test, False = full study
%cd /kaggle/working/pneumonia-detection
FLAG = '--quick' if QUICK else ''
!python -m src.pipeline $FLAG

## 3. Results

In [ ]:
%cd /kaggle/working/pneumonia-detection
from IPython.display import Image, Markdown, display
display(Image('artifacts/dataset_overview.png'))
for task in ['binary', 'multiclass']:
    display(Markdown(f'### Task: {task}'))
    display(Markdown(open(f'artifacts/{task}/results.md').read()))
    for f in ['model_comparison', 'roc_curves', 'reliability', 'referral_curve']:
        display(Image(f'artifacts/{task}/figures/{f}.png'))

## 4. Explainability (Grad-CAM) and confusion matrices

In [ ]:
import glob
for task in ['binary', 'multiclass']:
    for f in sorted(glob.glob(f'artifacts/{task}/figures/gradcam_*.png')) + sorted(glob.glob(f'artifacts/{task}/figures/confusion_*.png')):
        print(f); display(Image(f))

## 5. Package the artifacts
`artifacts.zip` (in Output) holds models, metrics and figures. Unzip it in the repo root to run the Streamlit app.

In [ ]:
!rm -rf artifacts/cache
!zip -qr /kaggle/working/artifacts.zip artifacts && ls -lh /kaggle/working/artifacts.zip